### **Script Teste de Geração**
##### Validar: Retrievel funcionando, Prompt + Contexto e Resposta com Rastreabilidade

In [3]:
import os
import sys
import contextlib
from typing import List, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field

import psycopg
from pgvector.psycopg import register_vector

from langchain_core.embeddings import Embeddings
from langchain_groq import ChatGroq

from FlagEmbedding import BGEM3FlagModel
from groq import Groq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

from pathlib import Path
import boto3

c:\Users\victo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### Carregar Variáveis de Ambiente

In [4]:
load_dotenv()

API_GRQ_LLM=os.getenv("API_KEY_GROQ_ESCREVEAI_VICTOR")
DATABASE_URL=os.getenv("DATABASE_URL")

#GROQ_GENERATION_MODEL = "llama-3.1-8b-instant"
GROQ_GENERATION_MODEL = "openai/gpt-oss-120b"

In [5]:
required_envs = {
    "API_KEY_GROQ_ESCREVEAI_VICTOR": API_GRQ_LLM,
    "DATABASE_URL": DATABASE_URL,
}

missing = [key for key, value in required_envs.items() if not value]

if missing:
    raise ValueError(f"Variáveis de ambiente ausentes: {missing}")

##### Estabelecer Conexões Aurora AWS e GROQ

In [6]:
# Estabeler conexão com o banco de dados
# Função para abrir conexão com o Aurora AWS
def abrir_conexao_pg():
    conn = psycopg.connect(
        DATABASE_URL,
        autocommit=False,
        keepalives=1,
        keepalives_idle=30,
        keepalives_interval=10,
        keepalives_count=5,
    )
    register_vector(conn)
    return conn

conn = abrir_conexao_pg()

# Estabelecer conexão com o S3
AWS_REGION = "us-east-1"
AWS_PROFILE = "victor.eduardo"

session = boto3.Session(
    profile_name=AWS_PROFILE,
    region_name=AWS_REGION,
)

s3_client = session.client("s3")
sts_client = session.client("sts")

# Estabelecer conexão com o Groq
client = Groq(api_key=API_GRQ_LLM)

#### **Modelo de Embedding do Input do Usuário**

In [7]:
class BGE_M3_Embeddings(Embeddings):
    def __init__(
        self,
        model_name: str = "BAAI/bge-m3",
        use_fp16: bool = True,
        batch_size: int = 12,
        max_length: int = 8192,
    ):
        self.model = BGEM3FlagModel(
            model_name,
            use_fp16=use_fp16,
        )
        self.batch_size = batch_size
        self.max_length = max_length

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        embeddings = self.model.encode(
            texts,
            batch_size=self.batch_size,
            max_length=self.max_length,
        )["dense_vecs"]

        return embeddings.tolist()

    def embed_query(self, text: str) -> List[float]:
        embedding = self.model.encode(
            [text],
            batch_size=1,
            max_length=self.max_length,
        )["dense_vecs"][0]

        return embedding.tolist()


# Modelo de embeddings da resposta do usuário da EscreveAI
modelo_baai_embeddings = BGE_M3_Embeddings(
    model_name="BAAI/bge-m3",
    use_fp16=True,
    batch_size=12,
    max_length=8192,
    )

Loading weights: 100%|██████████| 391/391 [00:07<00:00, 49.38it/s]


##### **Instânciar LLM de Geração**

In [8]:
# Modelo de LLM usado na geração da resposta do usuário
llm_groq = ChatGroq(
    model=GROQ_GENERATION_MODEL,
    temperature=0,
    api_key=API_GRQ_LLM,
)

In [9]:
# Função para transformar o briefing inteiro em uma consulta semântica única

def brief_para_consulta_rag(brief: dict) -> str:
    areas = ", ".join(brief.get("areasRelacionadas", []))

    if "Outro" in brief.get("areasRelacionadas", []):
        outro = brief.get("areasRelacionadasOutro", "").strip()
        if outro:
            areas += f", {outro}"

    partes_projetos = [
        f"Título do projeto: {brief.get('titulo', '')}",
        f"Áreas relacionadas: {areas}",
        f"Resumo do projeto: {brief.get('resumo', '')}",
        f"Objetivos: {brief.get('objetivos', '')}",
        f"Público-alvo: {brief.get('publicoAlvo', '')}",
        f"Metas, etapas e fases: {brief.get('metasEtapasFases', '')}",
        f"Entregáveis: {brief.get('entregaveis', '')}",
        f"Resultado esperado: {brief.get('resultadoEsperado', '')}",
        f"Duração/período: {brief.get('duracaoPeriodo', '')}",
    ]

    return "\n".join(
        parte for parte in partes_projetos
        if parte.split(":", 1)[1].strip()
    )

In [10]:
brief_usuario = {
    "titulo": "Capacitação em IA para professores",
    "areasRelacionadas": ["Educação", "Tecnologia"],
    "areasRelacionadasOutro": "",
    "resumo": "Projeto de formação continuada para professores da educação básica sobre uso pedagógico de inteligência artificial.",
    "objetivos": "Capacitar docentes para usar IA na criação de planos de aula, avaliação e personalização da aprendizagem.",
    "publicoAlvo": "Professores da educação básica da rede pública.",
    "metasEtapasFases": "Diagnóstico, oficinas práticas, acompanhamento pedagógico e avaliação final.",
    "entregaveis": "Materiais didáticos, trilha formativa, relatório de impacto e plano de continuidade.",
    "resultadoEsperado": "Professores capazes de aplicar ferramentas de IA de forma ética, crítica e produtiva.",
    "duracaoPeriodo": "12",
    "ragContextSource": "internal",
    "outputMode": "draft",
}

consulta_rag = brief_para_consulta_rag(brief_usuario)
print(consulta_rag)

Título do projeto: Capacitação em IA para professores
Áreas relacionadas: Educação, Tecnologia
Resumo do projeto: Projeto de formação continuada para professores da educação básica sobre uso pedagógico de inteligência artificial.
Objetivos: Capacitar docentes para usar IA na criação de planos de aula, avaliação e personalização da aprendizagem.
Público-alvo: Professores da educação básica da rede pública.
Metas, etapas e fases: Diagnóstico, oficinas práticas, acompanhamento pedagógico e avaliação final.
Entregáveis: Materiais didáticos, trilha formativa, relatório de impacto e plano de continuidade.
Resultado esperado: Professores capazes de aplicar ferramentas de IA de forma ética, crítica e produtiva.
Duração/período: 12


In [11]:
brief_usuario = {
    "titulo": "Restauração das fachadas da Faculdade de Medicina da UFRGS",
    "areasRelacionadas": ["Cultura", "Patrimônio Histórico", "Educação"],
    "areasRelacionadasOutro": "Intervenções em bens imóveis tombados/acautelados",
    "resumo": "Restauração das fachadas sul e noroeste do Antigo Prédio da Faculdade de Medicina da UFRGS, em Porto Alegre.",
    "objetivos": "Preservar o patrimônio histórico da UFRGS e recuperar elementos arquitetônicos das fachadas.",
    "publicoAlvo": "Comunidade acadêmica, escolas públicas e público geral.",
    "metasEtapasFases": "Planejamento, licitação, execução do restauro, ações educativas, divulgação e prestação de contas.",
    "entregaveis": "Fachadas restauradas, visitas guiadas, palestra/seminário, vídeo institucional e relatório final.",
    "resultadoEsperado": "Patrimônio restaurado, valorização da memória da UFRGS e ampliação do acesso à educação patrimonial.",
    "duracaoPeriodo": "12",
    "ragContextSource": "external",
    "outputMode": "draft",
}

consulta_rag = brief_para_consulta_rag(brief_usuario)


In [ ]:
brief_usuario = {
    "titulo": "Restauração das fachadas do Antigo Prédio da Faculdade de Medicina da UFRGS",
    "areasRelacionadas": ["Cultura", "Patrimônio Histórico", "Educação"],
    "areasRelacionadasOutro": "Intervenções em bens imóveis tombados/acautelados",
    "resumo": (
        "Projeto de restauração das fachadas sul e noroeste do Antigo Prédio da Faculdade "
        "de Medicina da UFRGS, em Porto Alegre, patrimônio histórico e cultural protegido "
        "nos âmbitos municipal, estadual e federal. A proposta busca recuperar elementos "
        "arquitetônicos ornamentais da edificação e ampliar o acesso público à memória "
        "histórica da universidade por meio de ações de educação patrimonial, visitas "
        "guiadas, palestra/seminário e produção audiovisual."
    ),
    "objetivos": (
        "Preservar um bem público de valor histórico, cultural e arquitetônico; restaurar "
        "as fachadas sul e noroeste do Antigo Prédio da Faculdade de Medicina da UFRGS; "
        "recuperar revestimentos, frisos, cornijas, elementos escultóricos, balaustradas "
        "e pintura; promover educação patrimonial; estimular o pertencimento social e "
        "fortalecer a valorização do patrimônio cultural edificado."
    ),
    "publicoAlvo": (
        "Comunidade acadêmica da UFRGS, estudantes de graduação, especialmente de "
        "Arquitetura e Urbanismo, professores e alunos de escolas públicas, pesquisadores, "
        "servidores, ex-alunos da Faculdade de Medicina e público geral interessado em "
        "patrimônio histórico e cultural."
    ),
    "metasEtapasFases": (
        "Pré-produção: elaboração e organização da documentação para licitação, alinhamento "
        "da equipe, produção de materiais gráficos, ações iniciais de comunicação e contato "
        "com possíveis palestrantes. Produção/execução: instalação do canteiro de obras, "
        "remoções e demolições necessárias, recuperação dos revestimentos, restauração de "
        "elementos ornamentais, pintura das fachadas, instalação de sistema de proteção "
        "para trabalho em altura, visitas técnicas, divulgação e realização de palestra/"
        "seminário. Pós-produção: divulgação da intervenção por banners informativos e "
        "elaboração do relatório de prestação de contas."
    ),
    "entregaveis": (
        "Fachadas sul e noroeste restauradas; visitas técnicas guiadas; atividades de "
        "educação patrimonial com escolas públicas; palestra ou seminário gratuito; "
        "materiais gráficos e de divulgação; vídeo de 3 a 7 minutos produzido pela UFRGS TV; "
        "banners informativos; relatório de prestação de contas."
    ),
    "resultadoEsperado": (
        "Recuperação material e simbólica de parte relevante do patrimônio histórico da "
        "UFRGS; ampliação do acesso público à história do prédio; fortalecimento da educação "
        "patrimonial; maior valorização da memória universitária e urbana de Porto Alegre; "
        "contribuição para a preservação de bens culturais brasileiros e para a cadeia "
        "produtiva da cultura local e regional."
    ),
    "duracaoPeriodo": "12",
    "ragContextSource": "external",
    "outputMode": "draft",
}

consulta_rag = brief_para_consulta_rag(brief_usuario)


In [38]:
brief_usuario = {
    "titulo": "Financiamento Adicional para o Projeto de Paz e Desenvolvimento",
    "areasRelacionadas": [
        "Educacao",
        "Agricultura",
        "Economia"
    ],
    "areasRelacionadasOutro": "Construção da paz, reintegração social e apoio a populações deslocadas",
    "resumo": "Financiamento adicional para consolidar o Projeto de Paz e Desenvolvimento na Colômbia, expandindo ações para Valle del Cauca e fortalecendo iniciativas locais de desenvolvimento e paz.",
    "objetivos": "Apoiar populações vulneráveis, de baixa renda e deslocadas em comunidades rurais e urbanas afetadas por conflito, reduzindo sua exposição à violência e promovendo reintegração social e econômica sustentável.",
    "publicoAlvo": "Populações vulneráveis, famílias de baixa renda, pessoas deslocadas pela violência, comunidades rurais e urbanas afetadas por conflito, organizações parceiras, unidades territoriais e comunidades indígenas.",
    "metasEtapasFases": "Expansão para Valle del Cauca, consolidação de subprojetos regionais, fortalecimento institucional de organizações parceiras e unidades territoriais, coordenação, avaliação, supervisão e estudos do projeto.",
    "entregaveis": "Subprojetos territoriais, apoio social, econômico, ambiental e comunitário, fortalecimento institucional, planos de desenvolvimento indígena, estudos, monitoramento e planos de mitigação ambiental.",
    "resultadoEsperado": "Consolidação da estratégia de construção da paz, melhoria da qualidade de vida de comunidades vulneráveis, reintegração de famílias deslocadas e fortalecimento da capacidade local de planejar, implementar e monitorar iniciativas de desenvolvimento e paz.",
    "duracaoPeriodo": "",
    "ragContextSource": "external",
    "outputMode": "draft",
}

##### **Validar Embedding**

In [39]:
consulta_rag = brief_para_consulta_rag(brief_usuario)

vetor_query = modelo_baai_embeddings.embed_query(consulta_rag)

print("---------------------------------")
print(" ---   VERIFICAR EMBEDDING    ---")
print("---------------------------------")
print(type(vetor_query))
print(len(vetor_query))
print(vetor_query[:5])
print("---------------------------------")

pre tokenize:   0%|          | 0/1 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.32it/s]

---------------------------------
 ---   VERIFICAR EMBEDDING    ---
---------------------------------
<class 'list'>
1024
[-0.02521209977567196, -0.0018204695079475641, -0.017695412039756775, -0.005432834383100271, -0.03281091898679733]
---------------------------------


##### **Verificar colunas e tipos das tabelas do banco de dados Aurora AWS**

In [40]:
# Verificar colunas das tabelas e tipos

with conn.cursor() as cur:
    cur.execute("""
        SELECT 
            table_schema,
            table_name,
            column_name,
            data_type,
            udt_name,
            ordinal_position
        FROM information_schema.columns
        WHERE table_schema = 'rag'
          AND table_name IN (
              'arquivos_documentos_rag',
              'chunks',
              'chunks_embeddings',
              'projetos_referencia_rag'
          )
        ORDER BY table_name, ordinal_position;
    """)

    colunas = cur.fetchall()

for row in colunas:
    print(row)

('rag', 'arquivos_documentos_rag', 'id', 'uuid', 'uuid', 1)
('rag', 'arquivos_documentos_rag', 'projetos_referencia_rag_id', 'uuid', 'uuid', 2)
('rag', 'arquivos_documentos_rag', 's3_bucket', 'character varying', 'varchar', 3)
('rag', 'arquivos_documentos_rag', 's3_key', 'text', 'text', 4)
('rag', 'arquivos_documentos_rag', 's3_versao_id', 'character varying', 'varchar', 5)
('rag', 'arquivos_documentos_rag', 'filename', 'character varying', 'varchar', 6)
('rag', 'arquivos_documentos_rag', 'mime_type', 'character varying', 'varchar', 7)
('rag', 'arquivos_documentos_rag', 'size_bytes', 'bigint', 'int8', 8)
('rag', 'arquivos_documentos_rag', 'sha256', 'character varying', 'varchar', 9)
('rag', 'arquivos_documentos_rag', 'processing_status', 'character varying', 'varchar', 10)
('rag', 'arquivos_documentos_rag', 'criado_em', 'timestamp with time zone', 'timestamptz', 12)
('rag', 'arquivos_documentos_rag', 'atualizado_em', 'timestamp with time zone', 'timestamptz', 13)
('rag', 'arquivos_docu

In [41]:
def buscar_chunks_similares(
    consulta: str,
    top_k: int = 12,
    dimensions: int = 1024,
    rag_context_source: str = "both",
):
    vetor = modelo_baai_embeddings.embed_query(consulta)

    if len(vetor) != dimensions:
        raise ValueError(
            f"Dimensão inválida do vetor da query: {len(vetor)}. "
            f"Esperado: {dimensions}."
        )

    filtros = [
        "ce.chunk_embedding IS NOT NULL",
        "ce.dimensions = %s",
    ]

    params_filtros = [dimensions]

    if rag_context_source == "internal":
        filtros.append("LOWER(TRIM(p.origin)) = LOWER(TRIM(%s))")
        params_filtros.append("Innovatis")

    elif rag_context_source == "external":
        filtros.append("LOWER(TRIM(p.origin)) <> LOWER(TRIM(%s))")
        params_filtros.append("Innovatis")

    elif rag_context_source == "both":
        pass

    else:
        raise ValueError(
            "rag_context_source inválido. Use: 'internal', 'external' ou 'both'."
        )

    where_clause = " AND ".join(filtros)

    sql = f"""
        SELECT
            c.id AS chunk_id,
            c.projetos_referencia_rag_id AS projeto_id,
            p.title_project AS projeto_titulo,
            p.area AS projeto_area,
            p.origin AS projeto_origem,
            p.status AS projeto_status,

            c.raw_text AS texto_chunk,
            c.page_start,
            c.page_end,
            c.chunk_index,
            c.token_count,
            c.char_count,
            c.source_origin,

            ce.model_embedding,
            ce.dimensions,

            1 - (ce.chunk_embedding <=> %s::vector) AS score_similaridade,
            ce.chunk_embedding <=> %s::vector AS distancia_cosseno

        FROM rag.chunks_embeddings ce

        JOIN rag.chunks c
            ON c.id = ce.chunk_id

        JOIN rag.projetos_referencia_rag p
            ON p.id = c.projetos_referencia_rag_id

        WHERE {where_clause}

        ORDER BY ce.chunk_embedding <=> %s::vector

        LIMIT %s;
    """

    params = [vetor, vetor] + params_filtros + [vetor, top_k]

    with conn.cursor() as cur:
        cur.execute(sql, params)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    return [dict(zip(columns, row)) for row in rows]

In [42]:
resultados = buscar_chunks_similares(
    consulta=consulta_rag,
    top_k=6,
    dimensions=1024,
    rag_context_source=brief_usuario["ragContextSource"],
)

print(len(resultados))

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]


6


In [43]:
# ============================================================
# 10. Visualização textual dos chunks retornados
# ============================================================

for i, r in enumerate(resultados, start=1):
    print("=" * 120)
    print(f"Resultado {i}")
    print(f"Projeto: {r['projeto_titulo']}")
    print(f"Área: {r['projeto_area']}")
    print(f"Origem: {r['projeto_origem']}")
    print(f"Chunk ID: {r['chunk_id']}")
    print(f"Projeto ID: {r['projeto_id']}")
    print(f"Chunk index: {r['chunk_index']}")
    print(f"Páginas: {r['page_start']} - {r['page_end']}")
    print(f"Token count: {r['token_count']}")
    print(f"Char count: {r['char_count']}")
    print(f"Modelo embedding: {r['model_embedding']}")
    print(f"Dimensões: {r['dimensions']}")
    print(f"Score similaridade: {r['score_similaridade']:.4f}")
    print(f"Distância cosseno: {r['distancia_cosseno']:.4f}")
    print()
    print(r["texto_chunk"][:300])
    print()

Resultado 1
Projeto: Financiamento Adicional para Desenvolvimento e Paz (IA)
Área: ['Social', 'Economia', 'Educacao']
Origem: World Bank
Chunk ID: bcb166ad-3ad9-4d36-8ea2-a6bc772827fb
Projeto ID: 88f7cdbf-ed00-4a96-bc24-5ce6b8da3f3c
Chunk index: 371
Páginas: 0 - 0
Token count: 518
Char count: 2644
Modelo embedding: BAAI/bge-m3
Dimensões: 1024
Score similaridade: 0.7150
Distância cosseno: 0.2850

PROJECT INFORMATION DOCUMENT (PID) 
CONCEPT STAGE 
Report No.:  AB4812 
Project Name Additional Financing for Peace and Development Project 
Region LATIN AMERICA AND CARIBBEAN 
Sector Other social services (30%); Sub-national government 
administration (20%); Adult literacy/non-formal education (20%

Resultado 2
Projeto: Financiamento Adicional para Desenvolvimento e Paz (IA)
Área: ['Social', 'Economia', 'Educacao']
Origem: World Bank
Chunk ID: 76c48c14-0d5d-4bd3-9942-b88b7151804e
Projeto ID: 88f7cdbf-ed00-4a96-bc24-5ce6b8da3f3c
Chunk index: 373
Páginas: 2 - 2
Token count: 318
Char count: 1550


In [44]:
def montar_contexto_rag(resultados: List[dict]) -> str:
    blocos = []

    for i, r in enumerate(resultados, start=1):
        texto = (r.get("texto_chunk") or "").strip()

        score = r.get("score_similaridade")
        score_fmt = f"{score:.4f}" if isinstance(score, (int, float)) else "N/A"

        bloco = f"""
[FONTE {i}]
projeto_id: {r.get("projeto_id")}
chunk_id: {r.get("chunk_id")}
projeto_titulo: {r.get("projeto_titulo")}
area: {r.get("projeto_area")}
origem: {r.get("projeto_origem")}
paginas: {r.get("page_start")} - {r.get("page_end")}
chunk_index: {r.get("chunk_index")}
score_similaridade: {score_fmt}

trecho:
{texto}
""".strip()

        blocos.append(bloco)

    return "\n\n" + ("\n\n" + "-" * 100 + "\n\n").join(blocos)

In [45]:
contexto_rag = montar_contexto_rag(resultados)
print(contexto_rag)



[FONTE 1]
projeto_id: 88f7cdbf-ed00-4a96-bc24-5ce6b8da3f3c
chunk_id: bcb166ad-3ad9-4d36-8ea2-a6bc772827fb
projeto_titulo: Financiamento Adicional para Desenvolvimento e Paz (IA)
area: ['Social', 'Economia', 'Educacao']
origem: World Bank
paginas: 0 - 0
chunk_index: 371
score_similaridade: 0.7150

trecho:
PROJECT INFORMATION DOCUMENT (PID) 
CONCEPT STAGE 
Report No.:  AB4812 
Project Name Additional Financing for Peace and Development Project 
Region LATIN AMERICA AND CARIBBEAN 
Sector Other social services (30%); Sub-national government 
administration (20%); Adult literacy/non-formal education (20%); 
Crops (20%); Other industry (10%) 
Project ID P101277 
Borrower(s) GOVERNMENT OF THE REPUBLIC OF COLOMBIA 
Implementing Agency Agencia Presidencial para la Acción Social y la Cooperación 
Internacional  
Environment Category []A [X] B   [ ] C   [ ] FI   [ ] TBD (to be determined) 
Date PID Prepared June 10, 2009 
Estimated Date of 
Appraisal Authorization 
June 22, 2009 
Estimated Date

PROMPT

In [46]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Você é um especialista em escrita de projetos institucionais.

Use o briefing do usuário e as fontes RAG recuperadas para gerar um documento estruturado.

Regras:
- Use as fontes RAG como referência, mas não copie literalmente.
- Não invente projetos de referência.
- Se alguma informação não estiver clara, faça uma inferência razoável.
- A saída deve ser JSON válido.
- Inclua os projetos e chunks usados como referência.
"""
    ),
    (
        "human",
        """
BRIEFING DO USUÁRIO:
{briefing}

FONTES RAG:
{contexto_rag}

MODO DE SAÍDA:
{modo_saida}

Gere o JSON com os seguintes campos:
- titulo
- area
- resumo
- objetivos
- publico_alvo
- metas_etapas_fases
- entregaveis
- resultado_esperado
- duracao_periodo
- projetos_referencia_usados
"""
    )
])

CHAIN

In [47]:
parser = JsonOutputParser()

chain = prompt | llm_groq | parser

In [48]:
gerar_projeto = chain.invoke({
    "briefing": consulta_rag,
    "contexto_rag": contexto_rag,
    "modo_saida": brief_usuario["outputMode"],
})

In [49]:
print(gerar_projeto)

{'titulo': 'Financiamento Adicional para o Projeto de Paz e Desenvolvimento', 'area': ['Educação', 'Agricultura', 'Economia'], 'resumo': 'Financiamento adicional destinado a consolidar e ampliar a estratégia de construção da paz já iniciada pelo Projeto de Paz e Desenvolvimento na Colômbia. O recurso permitirá a expansão das ações para o departamento de Valle del Cauca, a consolidação de sub‑projetos regionais e o fortalecimento institucional de organizações parceiras e unidades territoriais, garantindo a sustentabilidade de iniciativas de desenvolvimento local e de paz.', 'objetivos': ['Assistir populações vulneráveis, de baixa renda e deslocadas em comunidades rurais e urbanas afetadas pelo conflito, reduzindo sua exposição à violência.', 'Promover a reintegração social e econômica sustentável de famílias deslocadas por violência.', 'Consolidar a estratégia de construção da paz por meio de projetos territoriais integrados que combinam apoio social, econômico, ambiental e comunitário.

#### **Puxar projetos que são referência**

In [50]:
def baixar_documentos_referencia(
    conn,
    projeto_ids: list[str],
    pasta_destino: str = "referencias_projeto_doc",
):
    pasta = Path.cwd() / pasta_destino
    pasta.mkdir(parents=True, exist_ok=True)

    sql = """
        SELECT
            projetos_referencia_rag_id,
            s3_bucket,
            s3_key,
            filename
        FROM rag.arquivos_documentos_rag
        WHERE projetos_referencia_rag_id = ANY(%s)
    """

    with conn.cursor() as cur:
        cur.execute(sql, (projeto_ids,))
        arquivos = cur.fetchall()

    arquivos_baixados = []

    for projeto_id, bucket, key, filename in arquivos:

        caminho_local = (
            pasta /
            f"{projeto_id}_{filename}"
        )

        try:
            s3_client.download_file(
                Bucket=bucket,
                Key=key,
                Filename=str(caminho_local),
            )

            arquivos_baixados.append({
                "projeto_id": str(projeto_id),
                "filename": filename,
                "bucket": bucket,
                "key": key,
                "caminho_local": str(caminho_local),
                "status": "baixado",
            })

        except Exception as e:
            arquivos_baixados.append({
                "projeto_id": str(projeto_id),
                "filename": filename,
                "bucket": bucket,
                "key": key,
                "erro": str(e),
                "status": "erro",
            })

    return arquivos_baixados

In [51]:
def baixar_documentos_referencia(
    conn,
    projeto_ids: list[str],
    pasta_destino: str = "referencias_projeto_doc",
):
    pasta = Path.cwd() / pasta_destino
    pasta.mkdir(parents=True, exist_ok=True)

    print("Pasta destino:", pasta.resolve())

    projeto_ids = [str(pid) for pid in projeto_ids]

    sql = """
        SELECT
            projetos_referencia_rag_id,
            s3_bucket,
            s3_key,
            filename
        FROM rag.arquivos_documentos_rag
        WHERE projetos_referencia_rag_id = ANY(%s::uuid[])
          AND s3_bucket IS NOT NULL
          AND s3_key IS NOT NULL
    """

    with conn.cursor() as cur:
        cur.execute(sql, (projeto_ids,))
        arquivos = cur.fetchall()

    print("Arquivos encontrados no banco:", len(arquivos))

    arquivos_baixados = []

    for projeto_id, bucket, key, filename in arquivos:
        caminho_local = pasta / f"{projeto_id}_{Path(filename).name}"

        print("Tentando baixar:")
        print("Bucket:", bucket)
        print("Key:", key)
        print("Destino:", caminho_local)

        try:
            s3_client.download_file(
                Bucket=bucket,
                Key=key,
                Filename=str(caminho_local),
            )

            print("Baixado com sucesso:", caminho_local.exists())

            arquivos_baixados.append({
                "projeto_id": str(projeto_id),
                "filename": filename,
                "bucket": bucket,
                "key": key,
                "caminho_local": str(caminho_local),
                "status": "baixado",
            })

        except Exception as e:
            print("Erro no download:", repr(e))

            arquivos_baixados.append({
                "projeto_id": str(projeto_id),
                "filename": filename,
                "bucket": bucket,
                "key": key,
                "erro": repr(e),
                "status": "erro",
            })

    return arquivos_baixados

In [52]:
projeto_ids = list({
    r["projeto_id"]
    for r in resultados
})

In [53]:
projeto_ids = list({
    str(r["projeto_id"])
    for r in resultados
})

print("Total projeto_ids:", len(projeto_ids))
print(projeto_ids)

Total projeto_ids: 4
['3f9db30b-40d9-4779-9e86-989d5c1eec5e', '8cf0aa7a-9bc7-43df-9c68-f90ce412cd03', '88f7cdbf-ed00-4a96-bc24-5ce6b8da3f3c', '87ec2f8a-b067-4682-aa2c-8d5b33f0426b']


In [54]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            projetos_referencia_rag_id,
            s3_bucket,
            s3_key,
            filename
        FROM rag.arquivos_documentos_rag
        WHERE projetos_referencia_rag_id = ANY(%s::uuid[])
    """, (projeto_ids,))

    arquivos_debug = cur.fetchall()

print("Arquivos encontrados:", len(arquivos_debug))

for row in arquivos_debug:
    print(row)

Arquivos encontrados: 4
(UUID('88f7cdbf-ed00-4a96-bc24-5ce6b8da3f3c'), 'plataforma-escreveai', 'projetos-brutos/8dfa8380-99de-4c74-a65e-1af257b0a1cc/doc1.pdf', 'doc1.pdf')
(UUID('3f9db30b-40d9-4779-9e86-989d5c1eec5e'), 'plataforma-escreveai', 'projetos-brutos/6c120333-6d27-4feb-9a5e-e79522e30254/doc1.pdf', 'doc1.pdf')
(UUID('8cf0aa7a-9bc7-43df-9c68-f90ce412cd03'), 'plataforma-escreveai', 'projetos-brutos/8ef8a267-c463-4f72-a8bb-755949c96985/doc1.pdf', 'doc1.pdf')
(UUID('87ec2f8a-b067-4682-aa2c-8d5b33f0426b'), 'plataforma-escreveai', 'projetos-brutos/acd8fe43-ed6d-4290-b35c-3fce7e4f2b56/doc3.pdf', 'doc3.pdf')


In [55]:
arquivos_baixados = baixar_documentos_referencia(
    conn=conn,
    projeto_ids=projeto_ids,
)

Pasta destino: C:\Users\victo\OneDrive\Desktop\Arquivos Victor\Base_Doc_IA\Pipeline Geração\referencias_projeto_doc
Arquivos encontrados no banco: 4
Tentando baixar:
Bucket: plataforma-escreveai
Key: projetos-brutos/8dfa8380-99de-4c74-a65e-1af257b0a1cc/doc1.pdf
Destino: c:\Users\victo\OneDrive\Desktop\Arquivos Victor\Base_Doc_IA\Pipeline Geração\referencias_projeto_doc\88f7cdbf-ed00-4a96-bc24-5ce6b8da3f3c_doc1.pdf
Baixado com sucesso: True
Tentando baixar:
Bucket: plataforma-escreveai
Key: projetos-brutos/6c120333-6d27-4feb-9a5e-e79522e30254/doc1.pdf
Destino: c:\Users\victo\OneDrive\Desktop\Arquivos Victor\Base_Doc_IA\Pipeline Geração\referencias_projeto_doc\3f9db30b-40d9-4779-9e86-989d5c1eec5e_doc1.pdf
Baixado com sucesso: True
Tentando baixar:
Bucket: plataforma-escreveai
Key: projetos-brutos/8ef8a267-c463-4f72-a8bb-755949c96985/doc1.pdf
Destino: c:\Users\victo\OneDrive\Desktop\Arquivos Victor\Base_Doc_IA\Pipeline Geração\referencias_projeto_doc\8cf0aa7a-9bc7-43df-9c68-f90ce412cd03_d